In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install spatialdata "anndata==0.12.2" "scanpy==1.11.4" "squidpy==1.6.5"

## import

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    print("GPU: ",torch.cuda.get_device_name(0))
else:
    device = "cpu"
    print("Using CPU")

GPU:  Tesla T4


In [ ]:
##Working with google colab
import os
import sys

os.chdir("/content/drive/MyDrive/Thesis/Projects/Steamboat_X/examples_m_integrated_1_d")
cwd = os.getcwd()
print(cwd)

sys.path.append("../")

/content/drive/MyDrive/Thesis/Projects/Steamboat_X/examples_m_integrated_1_d


In [ ]:
import scanpy as sc
import squidpy as sq
from anndata import AnnData
import spatialdata as sd
import pandas as pd
from tqdm.notebook import tqdm
import scipy as sp
import numpy as np
import multiprocessing
import pickle as pkl
import torch
import gc
import sklearn.metrics
from sklearn.preprocessing import Normalizer, StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['mathtext.fontset'] = 'dejavuserif'
plt.rcParams['font.family'] = 'arial'

pltkw = dict(bbox_inches='tight', transparent=True)

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/local/lib/python3.12/dist-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [ ]:
import steamboat_m_integrated_1_d as sf
#importlib.reload(sm)
#import steamboat.integrated_model
# importlib.reload(spaceformer.benchmarks)

In [ ]:
import importlib
import steamboat_m_integrated_1_d.tools
import steamboat_m_integrated_1_d.model

## Creat Anndata

In [ ]:
Xenium_path = "/content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr"
sdata = sd.read_zarr(Xenium_path)
sdata

/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: Use

SpatialData object, with associated Zarr store: /content/drive/MyDrive/Thesis/Projects/Data/Mouse_Brain_Coronal/data.zarr
├── Images
│     ├── 'he_image': DataTree[cyx] (3, 24689, 17051), (3, 12344, 8525), (3, 6172, 4262), (3, 3086, 2131), (3, 1543, 1065)
│     └── 'morphology_focus': DataTree[cyx] (4, 23912, 34154), (4, 11956, 17077), (4, 5978, 8538), (4, 2989, 4269), (4, 1494, 2134)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
│     └── 'nucleus_labels': DataTree[yx] (23912, 34154), (11956, 17077), (5978, 8538), (2989, 4269), (1494, 2134)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (63173, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (63173, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (63036, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (63173, 5006)
with coordi

In [ ]:
adata = sdata.tables["table"]
adata

AnnData object with n_obs × n_vars = 63173 × 5006
    obs: 'cell_id', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'region', 'z_level', 'cell_labels'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatialdata_attrs'
    obsm: 'spatial'

In [ ]:
####SINA

adata.shape      # (cells, genes)
#adata.n_vars

(63173, 5006)

In [ ]:
adata_omiCLIP = sc.read_h5ad("../../Data/Mouse_Coronal_Embeddings/cells.h5ad")
adata_omiCLIP

AnnData object with n_obs × n_vars = 63173 × 1
    obs: 'cell_id'
    obsm: 'X_custom'

In [ ]:
# 1. Ensure cell IDs are the index (not just a column)
if 'cell_id' in adata.obs.columns:
    adata.obs.set_index('cell_id', inplace=True)
if 'cell_id' in adata_omiCLIP.obs.columns:
    adata_omiCLIP.obs.set_index('cell_id', inplace=True)

# 2. Align the two objects by cell_id (intersection)
common_ids = adata.obs_names.intersection(adata_omiCLIP.obs_names)

# Optional: check how many matched
print(f"Matched {len(common_ids)} cells out of {adata.n_obs}")

# 3. Reorder both to the same order
adata_c = adata[common_ids, :].copy()
adata_omiCLIP_c = adata_omiCLIP[common_ids, :].copy()

# 4. Add the X_custom matrix to adata_main.obsm
adata_c.obsm['Morpho_Embedding'] = adata_omiCLIP_c.obsm['X_custom']

# 5. Done! Verify
print(adata_c.obsm.keys())

Matched 63173 cells out of 63173
KeysView(AxisArrays with keys: spatial, Morpho_Embedding)


In [ ]:
adata_c.write("../../Data/Mouse_Coronal_Embeddings/Xenium_Morpho_integrated.h5ad")

## Train

In [ ]:
adata = sc.read_h5ad("../../Data/Breast_Cancer/ann_data.h5ad")

Founsation Models

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/h_optimus.h5ad")
M = edata.obsm['h_optimus']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/UNI.h5ad")
M = edata.obsm['UNI']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata = sc.read_h5ad("/content/drive/MyDrive/Thesis/Projects/Data/Breast_Cancer/FMs_3/virchow.h5ad")
M = edata.obsm['virchow']
ad = adata.copy()
ad.obsm["morpho"] = np.zeros_like(M)

In [ ]:
edata.obs_names = [str(int(cid[63:])-1) for cid in edata.obs['cell_id']]
common_cells = ad.obs_names.intersection(edata.obs_names)
ad.obsm['morpho'][ad.obs_names.get_indexer(common_cells)] = \
  edata.obsm['UNI'][edata.obs_names.get_indexer(common_cells)]

In [ ]:
#normalizer = Normalizer(norm="l2")
#ad.obsm['morpho'] = normalizer.transform(ad.obsm['morpho'])

adata.obsm['p_Morpho_Embedding'] = ad.obsm['morpho'] - np.min(ad.obsm['morpho'])

ROI Seperation

In [ ]:
edata = sc.read_h5ad("../../Data/Breast_Cancer/FMs_1/omiCLIP.h5ad")
N = edata.obsm['Omi_Clip'].astype(np.float32).copy()

In [ ]:
np.random.seed(42)

n = N.shape[1]
means = [0.0, 3.0, 6.0, 9.0]
std = 1.0

# generate vectors
v1 = np.random.normal(means[0], std, n)
v2 = np.random.normal(means[1], std, n)
v3 = np.random.normal(means[2], std, n)
v4 = np.random.normal(means[3], std, n)

# stack for joint normalization
V = np.vstack([v1, v2, v3, v4])

# normalize using global mean and std (preserves relative differences)
global_mean = V.mean()
global_std = V.std()

V_norm = (V - global_mean) / global_std

v1_norm, v2_norm, v3_norm, v4_norm = V_norm

In [ ]:
adata.obsm['p_Morpho_Embedding'] = np.zeros_like(N)
mask1 = adata.obs['annotation'] == 'DCIS_1'
mask2 = adata.obs['annotation'] == 'DCIS_2'
mask3 = adata.obs['annotation'] == 'Invasive'
mask4 = adata.obs['annotation'] == 'unassigned'
adata.obsm['p_Morpho_Embedding'][mask1] = v1_norm
adata.obsm['p_Morpho_Embedding'][mask2] = v2_norm
adata.obsm['p_Morpho_Embedding'][mask3] = v3_norm
adata.obsm['p_Morpho_Embedding'][mask4] = v4_norm

Cell_Type Separation

In [ ]:
E = np.random.normal(len(adata.obs['cell_type'].unique()), N.shape[1])
normalizer = Normalizer(norm="l2")
E = normalizer.transform(E)

In [ ]:
adata.obsm['p_Morpho_Embedding'] = np.zeros((adata.shape[0], N.shape[1]))
for (i, c_type) in enumerate(adata.obs['cell_type'].unique()):
    mask = adata.obs['cell_type'] == c_type
    adata.obsm['p_Morpho_Embedding'][mask] = E[i]

Noise

In [ ]:
E = np.random.normal(N.shape)
normalizer = Normalizer(norm="l2")
adata.obsm['p_Morpho_Embedding'] = normalizer.transform(E)

Preparing Dataset

In [ ]:
adatas = []
for i in adata.obs['region'].unique():
    adatas.append(adata[adata.obs['region'] == i])
    adatas[-1].obs['global'] = 0  #Only support one unique value for regional observation.

/tmp/ipython-input-4759537.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adatas[-1].obs['global'] = 0  #Only support one unique value for regional observation.


In [ ]:
adatas = sf.prep_adatas(adatas, norm=True, log1p=True)

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
dataset = sf.make_dataset(adatas, sparse_graph=True, regional_obs=['global'])

Using None to mask variables. Explicitly specify `mask_var=False` to use all genes.
Using ['global'] as regional annotations.


  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
print('Expression', dataset[0][0].shape)
print('Morpho_Embedding', dataset[0][1].shape)
print('Neighborhood_Graph', dataset[0][2].shape)
print('Regional_Expression', dataset[0][3][0].shape)
print('Regional_Morpho', dataset[0][4][0].shape)
print('Regional_Neighbors', dataset[0][5][0].shape)

Expression torch.Size([167780, 313])
Morpho_Embedding torch.Size([167780, 1536])
Neighborhood_Graph torch.Size([2, 1342240])
Regional_Expression torch.Size([1, 313])
Regional_Morpho torch.Size([1, 1536])
Regional_Neighbors torch.Size([2, 167780])


In [ ]:
importlib.reload(sf.dataset)
importlib.reload(sf.model)
importlib.reload(sf)

<module 'steamboat_m_integrated_1_d' from '/content/drive/MyDrive/Thesis/Projects/Steamboat_X/examples_m_integrated_1_d/../steamboat_m_integrated_1_d/__init__.py'>

In [ ]:
sf.set_random_seed(0)
model = sf.model.Steamboat(adatas[0].var_names.tolist(), Morpho_features = adata.obsm['p_Morpho_Embedding'].shape[1], n_heads=50, n_scales=3)
model = model.to(device)
# model.load_state_dict(torch.load('saved_models/mmbrain_new.pth', weights_only=True))


In [ ]:
model.fit(dataset, entry_masking_rate=0.3, feature_masking_rate=0.3,
          device=device,
          max_epoch=10000,
          loss_fun=torch.nn.MSELoss(reduction='mean'),
          opt=torch.optim.Adam,
          #sched=torch.optim.lr_scheduler.OneCycleLR,
          sched= None,
          max_lr=None, opt_args=dict(lr=0.01), stop_eps=1e-7, report_per=200, stop_tol=200)

DP koon


[2026-01-22 13:06:16,862::train::INFO] Epoch 1: train_loss 11.23581
INFO:train:Epoch 1: train_loss 11.23581
[2026-01-22 13:10:45,171::train::INFO] Epoch 201: train_loss 3.80259
INFO:train:Epoch 201: train_loss 3.80259
[2026-01-22 13:15:19,993::train::INFO] Epoch 401: train_loss 0.76367
INFO:train:Epoch 401: train_loss 0.76367
[2026-01-22 13:19:48,361::train::INFO] Epoch 601: train_loss 0.74149
INFO:train:Epoch 601: train_loss 0.74149
[2026-01-22 13:24:16,365::train::INFO] Epoch 801: train_loss 0.72607
INFO:train:Epoch 801: train_loss 0.72607
[2026-01-22 13:28:43,482::train::INFO] Epoch 1001: train_loss 0.71117
INFO:train:Epoch 1001: train_loss 0.71117
[2026-01-22 13:33:08,267::train::INFO] Epoch 1201: train_loss 0.68232
INFO:train:Epoch 1201: train_loss 0.68232
[2026-01-22 13:37:31,425::train::INFO] Epoch 1401: train_loss 0.67754
INFO:train:Epoch 1401: train_loss 0.67754
[2026-01-22 13:41:54,377::train::INFO] Epoch 1601: train_loss 0.66962
INFO:train:Epoch 1601: train_loss 0.66962
[202

Steamboat(
  (spatial_gather): BilinearAttention(
    (encoder_m): Encoder(
      (network): Sequential(
        (0): Linear(in_features=1536, out_features=1024, bias=True)
        (1): ReLU()
        (2): Linear(in_features=1024, out_features=256, bias=True)
        (3): ReLU()
        (4): Linear(in_features=256, out_features=313, bias=True)
        (5): ReLU()
      )
    )
    (q): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_Morpho): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_local): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (k_regionals): ModuleList(
      (0): NonNegLinear(
        (elu): ELU(alpha=1.0)
      )
    )
    (w_ego): NonNegScale(
      (elu): ELU(alpha=1.0)
    )
    (w_Morpho): NonNegScale(
      (elu): ELU(alpha=1.0)
    )
    (w_local): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (w_global): NonNegScale3(
      (elu): ELU(alpha=1.0)
    )
    (tanh): Tanh()
    (v): NonNegLinear(
      (elu): ELU(alpha=1.0)
    )
    (v_m):

In [ ]:
os.makedirs("saved_models", exist_ok=True)
if True:
    torch.save(model.state_dict(), 'saved_models/Xbcancer_50_0.3_UNI.pth')
else:
    model.load_state_dict(torch.load('saved_models/Xbcancer_50_0.3_UNI.pth',weights_only=True))